# How brittle is the MAD-Cars fine-tune? Query-corruption robustness

Post-hoc probe of LLM2CLIP EVA02-L-14-336 fold-0 OOF after MAD-Cars full pretrain → labeled fine-tune (`runs/cv/madcars_full_ft_ep35`, `current_best_tuned_madcars_full`). The **gallery stays clean**. Each query crop is replaced with `k` corrupted copies. We measure whether the embedding and the retrieved neighbors move.

Protocol: identity-disjoint val queries, cross-camera gallery, same embeddings as OOF scoring. No TTA, no postproc.

| Metric | Meaning | Robust if |
| --- | --- | --- |
| `cosine_mean` | mean cosine(clean query, each of `k` corrupted copies) | ~1 |
| `angular_mean_deg` | mean angle between those embeddings | ~0° |
| `pairwise_cosine` | the `k` copies agree with each other | ~1 |
| `overlap` / `jaccard` | top-10 gallery neighbors survive | ~1 |
| `kendall` | Kendall-τ on the union of the two top-10 lists | ~1 |
| `orig_r1_rank` | rank of the clean Rank-1 image after corruption | 1 |
| `rank1_id_same` | same vehicle still at Rank-1 | 1 |
| `delta_ap` | AP(corrupted) − AP(clean) on the clean gallery | ~0 |


In [1]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hydra import compose, initialize_config_dir
from PIL import Image

from augmentations import build_transforms
from dataset.images import crop_bbox, image_path
from eval import load_model
from posthoc import CORRUPTIONS, apply_k, compare_query, embed, overlap_curve, ranked_indices, summarize
from postproc.retrieval import normalize

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


def display(obj):
    print(obj if isinstance(obj, str) else obj.to_string() if hasattr(obj, "to_string") else obj)


CV = ROOT / "runs" / "cv" / "madcars_full_ft_ep35"
IMAGES = ROOT / "data" / "images"
FOLD = 0
VAL = CV / f"fold{FOLD}" / "val"
FIG_DIR = ROOT / "notebooks" / "eva02" / "posthoc_stability_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
README_FIG = ROOT / "notebooks" / "eva02" / "readme_figs"
README_FIG.mkdir(parents=True, exist_ok=True)
K = 3
K_NEIGH = 10
SEVERITIES = (1, 3, 5)
CORRUPTION_NAMES = [name for name in CORRUPTIONS if name != "identity"]


def pick_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    for index in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(index)
        if free / max(total, 1) > 0.95:
            return torch.device(f"cuda:{index}")
    return torch.device("cpu")


DEVICE = pick_device()
print("ROOT", ROOT)
print("device", DEVICE)
print("corruptions", CORRUPTION_NAMES)


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Please 'pip install xformers'
Please 'pip install xformers'


ROOT /data/projects/ryzhichkin/ReID
device cuda:0
corruptions ['gaussian_noise', 'impulse_noise', 'gaussian_blur', 'motion_blur', 'jpeg', 'brightness', 'contrast', 'saturate', 'downsample', 'occlude', 'rotate', 'crop', 'fog']


In [3]:
metrics = json.loads((VAL / "metrics.json").read_text())
ckpt = metrics["checkpoint"]
with initialize_config_dir(version_base="1.3", config_dir=str(ROOT / "configs")):
    cfg = compose(
        config_name="config",
        overrides=[
            f"checkpoint={ckpt}",
            "experiment=current_best_tuned_madcars_full",
            f"eval.device={DEVICE}",
            "eval.split=val",
            "eval.weights=ema",
            "eval.precision=fp32",
            f"eval.output_dir={VAL}",
            "eval.top_k=10",
            "eval.save_distances=false",
        ],
    )
model, cfg, checkpoint, weights = load_model(cfg, override_items=[])
model.to(DEVICE).eval()
CONTEXT = float(cfg.data.context_pct)
transform = build_transforms(cfg)
print("weights", weights, "pooling", cfg.model.pooling.kind, "size", list(cfg.data.image_size))


weights ema pooling attn size [336, 336]


In [4]:
oof = pd.read_csv(VAL / "oof.csv", dtype={"image_id": str})
query = pd.read_csv(VAL / "query.csv", dtype={"image_id": str})
gallery = pd.read_csv(VAL / "gallery.csv", dtype={"image_id": str})
emb = normalize(np.load(VAL / "embeddings.npy"))
lookup = {image_id: i for i, image_id in enumerate(oof.image_id)}
qe = emb[[lookup[i] for i in query.image_id]]
ge = emb[[lookup[i] for i in gallery.image_id]]
gids = gallery.vehicle_id.to_numpy()
gcams = gallery.camera_id.to_numpy()
qcams = query.camera_id.to_numpy()
dist = 1.0 - qe @ ge.T
rows = []
for i in range(len(query)):
    keep = gcams != qcams[i]
    order = np.argsort(dist[i], kind="stable")
    order = order[keep[order]]
    relevant = gids[order] == query.vehicle_id.iloc[i]
    found = np.flatnonzero(relevant)
    rows.append(
        {
            "q_index": i,
            "image_id": query.image_id.iloc[i],
            "vehicle_id": int(query.vehicle_id.iloc[i]),
            "ap": float(np.mean((np.arange(len(found)) + 1) / (found + 1))),
            "rank1_ok": bool(relevant[0]),
        }
    )
qtab = pd.DataFrame(rows)
easy = qtab.sort_values("ap", ascending=False).head(6)
hard = qtab.sort_values("ap").head(6)
picked = pd.concat([easy, hard]).drop_duplicates("image_id").reset_index(drop=True)
print(qtab.ap.describe())
display(picked[["image_id", "vehicle_id", "ap", "rank1_ok"]])

count    309.000000
mean       0.900440
std        0.228248
min        0.011509
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: ap, dtype: float64
                            image_id  vehicle_id        ap  rank1_ok
0   f8100578c7344b06acd040ae4ea6543a        1283  1.000000      True
1   34ce7989a4c440a882c40fa7891ce9d4          63  1.000000      True
2   d356962e487547a193f2939fb910b202        1249  1.000000      True
3   b7a66b2051464dc3b66dd5fab7a5e85d        1488  1.000000      True
4   cfcbf7df946d44069d1c1ef6ec4f0d6b          78  1.000000      True
5   3b54aa0d64dc4176be41316a7a81db57        1527  1.000000      True
6   f368431f36ab443ebe920d0059219abb         208  0.011509     False
7   9566fa2e6c2b404e8a4301424d8e333e         485  0.023366     False
8   2921a3fdaff34be18d0018b89f2e37e9         590  0.052931     False
9   e8a9a22033b6465f8095e7408d56e950        1278  0.055742     False
10  1aaa9ef342274bb7aaf54dcf652dae0c         670  0.0947

In [5]:
def frame_row(image_id):
    hit = oof.loc[oof.image_id == image_id]
    if len(hit) == 0:
        hit = pd.concat([query, gallery], ignore_index=True)
        hit = hit.loc[hit.image_id == image_id]
    return hit.iloc[0]


def load_crop_array(image_id):
    row = frame_row(image_id)
    with Image.open(image_path(IMAGES, image_id)) as im:
        crop = crop_bbox(im.convert("RGB"), [row.x, row.y, row.w, row.h], CONTEXT)
    return np.asarray(crop.convert("RGB")), row


def keep_mask(q_index):
    return gcams != qcams[q_index]


def embed_crops(crops):
    tensors = torch.stack([transform(np.asarray(crop)) for crop in crops])
    return embed(model, tensors, device=DEVICE, batch_size=8)


print("helpers ready")

helpers ready


In [6]:
id_rows = []
for rec in picked.to_dict("records"):
    rgb, _ = load_crop_array(rec["image_id"])
    live = embed_crops([rgb])[0]
    stored = qe[rec["q_index"]]
    crops = apply_k(rgb, "identity", severity=3, k=2, seed=0)
    corr = embed_crops(crops)
    row = compare_query(
        stored,
        corr,
        ge,
        qid=rec["vehicle_id"],
        gids=gids,
        keep=keep_mask(rec["q_index"]),
        k_neighbors=K_NEIGH,
    )
    row["live_cosine"] = float(np.clip(live @ stored, -1, 1))
    id_rows.append(row)
id_tab = pd.DataFrame(id_rows)
display(
    id_tab[["cosine_mean", "overlap", "delta_ap", "orig_r1_rank", "live_cosine"]]
    .describe()
    .loc[["mean", "min"]]
)
if id_tab.cosine_mean.min() < 0.999 or id_tab.live_cosine.min() < 0.999:
    raise RuntimeError("Identity / re-embed control drifted from OOF embeddings")
print("identity control ok")

      cosine_mean  overlap  delta_ap  orig_r1_rank  live_cosine
mean     0.999916      1.0  0.000925           1.0     0.999914
min      0.999784      1.0 -0.000634           1.0     0.999770
identity control ok


In [7]:
records = []
n_jobs = len(picked) * len(CORRUPTION_NAMES) * len(SEVERITIES)
done = 0
for rec in picked.to_dict("records"):
    rgb, _ = load_crop_array(rec["image_id"])
    keep = keep_mask(rec["q_index"])
    for name in CORRUPTION_NAMES:
        for severity in SEVERITIES:
            crops = apply_k(rgb, name, severity=severity, k=K, seed=1000 + rec["q_index"] * 17 + severity)
            corr = embed_crops(crops)
            row = compare_query(
                qe[rec["q_index"]],
                corr,
                ge,
                qid=rec["vehicle_id"],
                gids=gids,
                keep=keep,
                k_neighbors=K_NEIGH,
                orig_image=rgb,
                corr_images=crops,
                include_samples=False,
            )
            row.update(
                {
                    "corruption": name,
                    "severity": severity,
                    "image_id": rec["image_id"],
                    "q_index": rec["q_index"],
                    "ap_clean": rec["ap"],
                    "rank1_ok": rec["rank1_ok"],
                }
            )
            records.append(row)
            done += 1
            if done % 40 == 0 or done == n_jobs:
                print(f"{done}/{n_jobs}")
probe = pd.DataFrame(records)
print("rows", len(probe))

40/468


80/468


120/468


160/468


200/468


...

320/468


360/468


400/468


440/468


468/468
rows 468


In [8]:
summary = pd.DataFrame(summarize(records))
order = list(pd.Series(summary.groupby("corruption")["cosine_mean"].mean()).sort_values().index)
summary["corruption"] = pd.Categorical(summary["corruption"], order, ordered=True)
summary = summary.sort_values(["corruption", "severity"])
cols = [
    "corruption",
    "severity",
    "n",
    "cosine_mean",
    "angular_mean_deg",
    "overlap",
    "jaccard",
    "kendall",
    "orig_r1_rank",
    "rank1_id_same",
    "delta_ap",
    "l1",
]
display(summary[cols].round(3))
print(
    "worst cosine",
    summary.sort_values("cosine_mean")[["corruption", "severity", "cosine_mean", "overlap", "delta_ap"]]
    .head(8)
    .to_string(index=False),
)
print(
    "worst overlap",
    summary.sort_values("overlap")[["corruption", "severity", "cosine_mean", "overlap", "delta_ap"]]
    .head(8)
    .to_string(index=False),
)

        corruption  severity    n  cosine_mean  angular_mean_deg  overlap  jaccard  kendall  orig_r1_rank  rank1_id_same  delta_ap     l1
33            crop       1.0  3.0        0.974            12.397    0.869    0.779    0.720         1.111          0.972    -0.001  0.180
34            crop       3.0  3.0        0.901            24.663    0.761    0.645    0.509         2.194          0.750     0.011  0.246
35            crop       5.0  3.0        0.564            53.914    0.444    0.334    0.067        67.056          0.611    -0.103  0.264
18        contrast       1.0  3.0        0.980            10.883    0.894    0.826    0.786         1.056          1.000     0.004  0.044
19        contrast       3.0  3.0        0.921            21.835    0.794    0.686    0.586         1.944          0.917     0.005  0.130
20        contrast       5.0  3.0        0.877            28.200    0.694    0.552    0.432         2.861          0.861     0.007  0.171
36             fog       1.0  3.0 

In [9]:
def heatmap(frame, value, title, path, cmap="magma", vmin=None, vmax=None, fmt="{:.2f}"):
    table = frame.pivot_table(index="corruption", columns="severity", values=value, observed=True)
    table = table.loc[order]
    fig, ax = plt.subplots(figsize=(4.8, 6.2))
    im = ax.imshow(table.to_numpy(), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(table.columns)), [str(c) for c in table.columns])
    ax.set_yticks(range(len(table.index)), list(table.index))
    ax.set_xlabel("severity")
    ax.set_title(title)
    for r, name in enumerate(table.index):
        for c, sev in enumerate(table.columns):
            ax.text(c, r, fmt.format(table.loc[name, sev]), ha="center", va="center", color="w", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()


heatmap(
    summary,
    "cosine_mean",
    "Embedding cosine (clean vs corrupted)",
    FIG_DIR / "cosine.png",
    vmin=0.4,
    vmax=1.0,
)
Image.open(FIG_DIR / "cosine.png").convert("RGB").save(README_FIG / "posthoc_cosine.jpg", quality=92)
heatmap(summary, "overlap", "Top-10 neighbor overlap", FIG_DIR / "overlap.png", vmin=0.0, vmax=1.0)
heatmap(
    summary,
    "delta_ap",
    "ΔAP (corrupted − clean)",
    FIG_DIR / "delta_ap.png",
    cmap="coolwarm",
    vmin=-0.4,
    vmax=0.05,
    fmt="{:.2f}",
)


Saved heatmaps:

![cosine](posthoc_stability_figs/cosine.png)

![overlap](posthoc_stability_figs/overlap.png)

![delta AP](posthoc_stability_figs/delta_ap.png)


In [10]:
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for name, grp in probe.groupby("corruption"):
    ax.scatter(grp.l1, grp.cosine_mean, s=18, alpha=0.75, label=name)
ax.set_xlabel("pixel L1 (crop vs corrupted)")
ax.set_ylabel("embedding cosine")
ax.set_title("Sensitivity: pixel shift vs embedding move")
ax.legend(ncol=2, fontsize=8, frameon=False, loc="lower left")
fig.tight_layout()
fig.savefig(FIG_DIR / "sensitivity.png", dpi=120, bbox_inches="tight")
fig.savefig(README_FIG / "posthoc_sensitivity.jpg", dpi=120, bbox_inches="tight")
plt.show()


![sensitivity](posthoc_stability_figs/sensitivity.png)


In [11]:
curve_names = ["gaussian_noise", "jpeg", "occlude", "crop", "motion_blur"]
curve_rec = picked.iloc[0]
rgb, _ = load_crop_array(curve_rec.image_id)
keep = keep_mask(curve_rec.q_index)
orig_order = ranked_indices(ge @ qe[curve_rec.q_index], keep)
ks = (1, 5, 10, 20, 50)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for name in curve_names:
    crop = apply_k(rgb, name, severity=5, k=1, seed=7)[0]
    corr = embed_crops([crop])[0]
    corr_order = ranked_indices(ge @ corr, keep)
    curve = overlap_curve(orig_order, corr_order, ks=ks)
    ax.plot(list(curve), list(curve.values()), marker="o", label=name)
ax.set_xlabel("k")
ax.set_ylabel("neighbor overlap@k")
ax.set_title(f"Overlap vs k · query {curve_rec.image_id} · severity 5")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "overlap_k.png", dpi=120, bbox_inches="tight")
plt.show()

![overlap vs k](posthoc_stability_figs/overlap_k.png)


In [12]:
show_names = ["identity", "jpeg", "gaussian_noise", "motion_blur", "occlude", "crop", "fog"]
show_rec = picked.iloc[0]
rgb, _ = load_crop_array(show_rec.image_id)
fig, axes = plt.subplots(len(show_names), 1 + K, figsize=(2.2 * (1 + K), 2.1 * len(show_names)))
for r, name in enumerate(show_names):
    axes[r, 0].imshow(rgb)
    axes[r, 0].set_ylabel(name, fontsize=9)
    axes[r, 0].set_xticks([])
    axes[r, 0].set_yticks([])
    if r == 0:
        axes[r, 0].set_title("clean", fontsize=9)
    crops = apply_k(rgb, name, severity=5, k=K, seed=11)
    for c in range(K):
        axes[r, c + 1].imshow(crops[c])
        axes[r, c + 1].axis("off")
        if r == 0:
            axes[r, c + 1].set_title(f"k={c + 1}", fontsize=9)
fig.suptitle(f"k={K} copies at severity 5 · query {show_rec.image_id}", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "corruptions.png", dpi=120, bbox_inches="tight")
plt.show()

![corruptions](posthoc_stability_figs/corruptions.png)


In [13]:
case = probe.sort_values("delta_ap").iloc[0]
rgb, _ = load_crop_array(case.image_id)
keep = keep_mask(int(case.q_index))
orig_sim = ge @ qe[int(case.q_index)]
orig_order = ranked_indices(orig_sim, keep)
crop = apply_k(rgb, case.corruption, severity=int(case.severity), k=1, seed=13)[0]
corr = embed_crops([crop])[0]
corr_order = ranked_indices(ge @ corr, keep)
n_show = 5


def gallery_crop(idx):
    image_id = gallery.image_id.iloc[int(idx)]
    pic, _ = load_crop_array(image_id)
    return pic, image_id, int(gids[int(idx)])


fig, axes = plt.subplots(2, n_show + 1, figsize=(2.1 * (n_show + 1), 4.6))
axes[0, 0].imshow(rgb)
axes[0, 0].set_title("clean query", fontsize=8)
axes[0, 0].axis("off")
axes[1, 0].imshow(crop)
axes[1, 0].set_title(f"{case.corruption} s{int(case.severity)}", fontsize=8)
axes[1, 0].axis("off")
for i in range(n_show):
    pic, image_id, vid = gallery_crop(orig_order[i])
    axes[0, i + 1].imshow(pic)
    axes[0, i + 1].set_title(f"clean #{i + 1} id {vid}", fontsize=8)
    axes[0, i + 1].axis("off")
    pic, image_id, vid = gallery_crop(corr_order[i])
    axes[1, i + 1].imshow(pic)
    axes[1, i + 1].set_title(f"corr #{i + 1} id {vid}", fontsize=8)
    axes[1, i + 1].axis("off")
fig.suptitle(
    f"query {case.image_id}  ΔAP {case.delta_ap:.3f}"
    f"  overlap@10 {case.overlap:.2f}  cosine {case.cosine_mean:.3f}",
    y=1.02,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "neighbors.png", dpi=120, bbox_inches="tight")
plt.show()
print(
    case[
        ["corruption", "severity", "cosine_mean", "overlap", "orig_r1_rank", "delta_ap", "rank1_id_same"]
    ].to_string()
)

corruption            crop
severity                 5
cosine_mean       0.461255
overlap           0.533333
orig_r1_rank     71.333333
delta_ap         -0.324614
rank1_id_same     0.666667


![neighbors](posthoc_stability_figs/neighbors.png)


## How to read this

- **Identity control** must stay at cosine ≈ 1 and overlap = 1. If it does not, the probe is not using the OOF embedding.
- **`crop` / `occlude` / `downsample`** remove evidence the model needs (panel, lights, grille). Neighbor overlap dropping here means the embedding is using those parts, not a global color blob — consistent with the `interp/` maps.
- **`jpeg` / `gaussian_noise` / `motion_blur`** are closer to real cameras. Cosine can stay high while Rank-1 identity flips: the vector moves a little, but that is enough to swap lookalikes.
- **`brightness` / `contrast` / `fog`** test color constancy. A large angular move with small pixel L1 is worse than a large L1 with a small angular move.
- `pairwise_cosine` near 1 on a stochastic corruption (noise, occlude) means the embedder collapses those samples; a drop means each draw lands somewhere else.
- This is a query-side probe. Gallery images stay clean, so it does not measure transductive postproc robustness.
